In [3]:
import lightgbm as lgb
import pandas as pd
import optuna
import warnings
import json

from SharedModules import input_dir, model_dir
from SharedModules.tuner import tune_lgbm_params
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

_train = pd.read_csv(input_dir + "train.csv", index_col=0)
_test = pd.read_csv(input_dir + "test.csv", index_col=0)

target = "Heart Disease"

cv = KFold(n_splits=10, shuffle=True, random_state=42)

X = _train.drop(target, axis=1)
y = _train[target].map({"Absence":0, "Presence": 1})
X_test = _test

In [4]:
def objective(trial):
    param = tune_lgbm_params(trial)

    cv = KFold(n_splits=10, shuffle=True, random_state=42)
    aucs = []

    for train_idx, valid_idx in cv.split(X, y):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        model = lgb.LGBMClassifier(**param, verbose=-1)

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric="auc",
            # categorical_feature=cat_cols,  # Pass categorical columns
            callbacks=[lgb.early_stopping(50)],  # <-- pruning
        )

        preds = model.predict_proba(X_valid)[:, 1]
        aucs.append(roc_auc_score(y_valid, preds))

    return sum(aucs) / len(aucs)


# Run the tuner
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        multivariate=True,
        group=True,
        n_startup_trials=30,  # random exploration first
        seed=42,
    ),
    # pruner=optuna.pruners.HyperbandPruner(
    #     n_startup_trials=50,  # let Optuna learn first
    #     n_warmup_steps=300,  # no pruning before 300 trees
    #     interval_steps=50,  # check every 50 trees
    # ),
    study_name="my_lgbm_study",
    storage=f"sqlite:///{model_dir}optuna_S6E02.db",
    load_if_exists=True,
)
study.optimize(objective, n_trials=300)

print("Best params:", study.best_params)
print("Best AUC:", study.best_value)


[I 2026-02-03 22:49:34,108] Using an existing study with name 'my_lgbm_study' instead of creating a new one.


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1702]	valid_0's auc: 0.955546
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1401]	valid_0's auc: 0.954732
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1263]	valid_0's auc: 0.955147
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1089]	valid_0's auc: 0.955333
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1543]	valid_0's auc: 0.954503
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1580]	valid_0's auc: 0.957372
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1346]	valid_0's auc: 0.954822
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1265]	valid_0'

[I 2026-02-03 22:58:27,568] Trial 22 finished with value: 0.9552039928574768 and parameters: {'learning_rate': 0.01184431975182039, 'n_estimators': 4828, 'num_leaves': 102, 'max_depth': 8, 'min_child_samples': 21, 'min_child_weight': 0.004207053950287938, 'min_split_gain': 0.05808361216819946, 'subsample': 0.9464704583099741, 'subsample_freq': 7, 'feature_fraction': 0.8832290311184181, 'reg_alpha': 1.4610865886287176e-08, 'reg_lambda': 2.7366339528977597, 'max_bin': 223, 'scale_pos_weight': 1.2671460434922075}. Best is trial 20 with value: 0.955473990942043.


Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2133]	valid_0's auc: 0.955609
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2142]	valid_0's auc: 0.954745
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1983]	valid_0's auc: 0.955266
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2142]	valid_0's auc: 0.955496
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2133]	valid_0's auc: 0.954548
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2142]	valid_0's auc: 0.957452
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2128]	valid_0's auc: 0.954936
Training until validation scores 

[I 2026-02-03 23:08:40,814] Trial 23 finished with value: 0.9553032998235672 and parameters: {'learning_rate': 0.007599674150654906, 'n_estimators': 2142, 'num_leaves': 60, 'max_depth': 7, 'min_child_samples': 40, 'min_child_weight': 0.014618962793704957, 'min_split_gain': 0.6118528947223795, 'subsample': 0.6557975442608167, 'subsample_freq': 3, 'feature_fraction': 0.7465447373174767, 'reg_alpha': 4.452048365748842e-05, 'reg_lambda': 0.06764288719681069, 'max_bin': 102, 'scale_pos_weight': 1.9313157645099457}. Best is trial 20 with value: 0.955473990942043.


...

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2640]	valid_0's auc: 0.955886
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2217]	valid_0's auc: 0.954923
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2071]	valid_0's auc: 0.955553
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1761]	valid_0's auc: 0.955714
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2640]	valid_0's auc: 0.9548
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2654]	valid_0's auc: 0.957683
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2657]	valid_0's auc: 0.955162
Training until validation scores don't improve for 50 rounds


[I 2026-02-04 10:37:48,147] Trial 151 finished with value: 0.9555365989849642 and parameters: {'learning_rate': 0.023530625556260152, 'n_estimators': 2657, 'num_leaves': 111, 'max_depth': 4, 'min_child_samples': 73, 'min_child_weight': 0.18404590532562845, 'min_split_gain': 0.7229069708663247, 'subsample': 0.9714417673645673, 'subsample_freq': 1, 'feature_fraction': 0.7177601511059486, 'reg_alpha': 5.034181856399477e-07, 'reg_lambda': 6.232792873336246e-07, 'max_bin': 192, 'scale_pos_weight': 2.3799705691857675}. Best is trial 56 with value: 0.9555838955195093.


Training until validation scores don't improve for 50 rounds


[W 2026-02-04 10:38:03,993] Trial 152 failed with parameters: {'learning_rate': 0.013234126967940019, 'n_estimators': 4858, 'num_leaves': 53, 'max_depth': 5, 'min_child_samples': 20, 'min_child_weight': 2.030972279811903, 'min_split_gain': 0.8745198916330421, 'subsample': 0.8325644826094621, 'subsample_freq': 2, 'feature_fraction': 0.6027740063389871, 'reg_alpha': 4.431397088384314e-05, 'reg_lambda': 0.1934178031801781, 'max_bin': 196, 'scale_pos_weight': 0.9908079099269849} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\Blanc\DataScientist\Kaggle\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\Blanc\AppData\Local\Temp\ipykernel_26024\1560624575.py", line 13, in objective
    model.fit(
  File "c:\Users\Blanc\DataScientist\Kaggle\.venv\Lib\site-packages\lightgbm\sklearn.py", line 1560, in fit
    super().fit(
  File "

KeyboardInterrupt: 

In [5]:
best_params = study.best_params
with open(model_dir + "lgbm_base_params.json", "w") as f:
    json.dump(best_params, f, indent=4)